In [1]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt

from wordcloud import wordcloud
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\pranto\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\pranto\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [2]:
df= pd.read_csv('spam.csv')
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [6]:
df.drop(columns=['Unnamed: 2','Unnamed: 3','Unnamed: 4'],inplace=True)
df.head()

,v1,v2
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [8]:
df.rename(columns={'v1':'target','v2':'text'},inplace=True)
df.head()

,target,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [9]:
from sklearn.preprocessing import LabelEncoder

encoder=LabelEncoder()
df['target']=encoder.fit_transform(df['target'])

df.head()

,target,text
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


In [10]:
df.duplicated().sum()

np.int64(403)

In [11]:
len(df)

5572

In [12]:
df=df.drop_duplicates(keep='first')
len(df)

5169

Feature Engineering

In [28]:
from nltk.tokenize import TreebankWordTokenizer
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
import string
import nltk

# Ensure stopwords are downloaded
nltk.download('stopwords')

ps = PorterStemmer()
tokenizer = TreebankWordTokenizer()
stop_words = set(stopwords.words('english'))


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\pranto\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [30]:
def transform_text(text):
    text = text.lower()
    text = tokenizer.tokenize(text)  # Use Treebank tokenizer

    y = []
    # Keep alphanumeric words only
    for i in text:
        if i.isalnum():
            y.append(i)

    text = y[:]
    y.clear()

    # Remove stopwords and punctuation
    for i in text:
        if i not in stop_words and i not in string.punctuation:
            y.append(i)

    text = y[:]
    y.clear()

    # Stem words
    for i in text:
        y.append(ps.stem(i))

    return " ".join(y)

In [32]:
transform_text('Go an do.')

'go'

In [33]:
df['transformed_text']=df["text"].apply(transform_text)
df.head()

,target,text,transformed_text
0,0,"Go until jurong point, crazy.. Available only ...",go jurong point avail bugi n great world la e ...
1,0,Ok lar... Joking wif u oni...,ok lar joke wif u oni
2,1,Free entry in 2 a wkly comp to win FA Cup fina...,free entri 2 wkli comp win fa cup final tkt 21...
3,0,U dun say so early hor... U c already then say...,u dun say earli hor u c alreadi say
4,0,"Nah I don't think he goes to usf, he lives aro...",nah think goe usf live around though


In [34]:
from sklearn.feature_extraction.text import CountVectorizer , TfidfVectorizer

tfid= TfidfVectorizer(max_features=500)

In [36]:
x=tfid.fit_transform(df['transformed_text']).toarray()
y=df['target'].values

train and test split

In [45]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.20,random_state=2)

In [56]:
from xgboost import XGBClassifier

# Initialize your classifier
xgb = XGBClassifier(n_estimators=50, random_state=2)

# Put it in a dictionary
clf = {
    'xgb': xgb
}

from sklearn.metrics import accuracy_score, precision_score

def train_classifier(clfs, x_train,y_train,x_test,y_test):
    clfs.fit(x_train,y_train)
    y_pred=clfs.predict(x_test)
    accuracy=accuracy_score(y_test,y_pred)
    precision=precision_score(y_test,y_pred)
    return accuracy, precision

# Now you can loop
accuracy_list = []
precision_list = []

for name, model in clf.items():
    current_accuracy, current_precision = train_classifier(model, x_train, y_train, x_test, y_test)
    
    print(f"\nFor: {name}")
    print("Accuracy:", current_accuracy)
    print("Precision:", current_precision)
    
    accuracy_list.append(current_accuracy)
    precision_list.append(current_precision)



For: xgb
Accuracy: 0.9661508704061895
Precision: 0.963963963963964
